# Credit Scoring — Fase 4: Modelado Avanzado

Entrenamos y optimizamos los modelos de boosting más potentes:  
- **XGBoost** con Optuna (100 trials + pruning)  
- **LightGBM** con early stopping  
- **CatBoost** con manejo nativo de categoricals  

Al final seleccionamos el **modelo campeón** por AUC + interpretabilidad.

In [ ]:
import sys
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from pathlib import Path

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

from sklearn.metrics import (
    roc_auc_score, roc_curve, brier_score_loss
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sys.path.append('..')

DATA_PROC = Path('../data/processed')
MODELS    = Path('../models/saved')
FIGURES   = Path('../reports/figures')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Carga de datos

In [ ]:
train = pd.read_csv(DATA_PROC / 'train_raw.csv')
val   = pd.read_csv(DATA_PROC / 'val_raw.csv')
test  = pd.read_csv(DATA_PROC / 'test_raw.csv')

X_train, y_train = train.drop('target', axis=1), train['target']
X_val,   y_val   = val.drop('target', axis=1),   val['target']
X_test,  y_test  = test.drop('target', axis=1),  test['target']

# Scale pos weight para XGBoost (ratio negativos/positivos)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Train: {X_train.shape} | scale_pos_weight: {scale_pos_weight:.1f}')

## 2. Funciones auxiliares

In [ ]:
def ks_statistic(y_true, y_prob) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(np.max(tpr - fpr))


def score_model(name: str, model, X_val, y_val) -> dict:
    y_prob = model.predict_proba(X_val)[:, 1]
    auc    = roc_auc_score(y_val, y_prob)
    return {
        'model':    name,
        'AUC-ROC':  auc,
        'KS':       ks_statistic(y_val, y_prob),
        'Gini':     2 * auc - 1,
        'Brier':    brier_score_loss(y_val, y_prob),
        '_y_prob':  y_prob,
    }

print('OK')

## 3. XGBoost + Optuna (100 trials)

Optuna usa **TPE sampler** con **MedianPruner** para eliminar trials malos temprano.  
Optimizamos AUC-ROC en el set de validación.

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

def xgb_objective(trial: optuna.Trial) -> float:
    params = {
        'objective':        'binary:logistic',
        'eval_metric':      'auc',
        'scale_pos_weight': scale_pos_weight,
        'verbosity':        0,
        'seed':             42,
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
    }

    n_estimators = params.pop('n_estimators')
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=n_estimators,
        evals=[(dval, 'val')],
        early_stopping_rounds=30,
        verbose_eval=False,
    )
    y_prob = model.predict(dval)
    return roc_auc_score(y_val, y_prob)


study_xgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
)
study_xgb.optimize(xgb_objective, n_trials=100, show_progress_bar=True)

print(f'\nMejor AUC XGBoost: {study_xgb.best_value:.4f}')
print(f'Mejores params: {study_xgb.best_params}')

In [ ]:
# Reentrenar con los mejores hiperparámetros
best_xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'scale_pos_weight': scale_pos_weight,
    'verbosity': 0,
    'seed': 42,
    **{k: v for k, v in study_xgb.best_params.items() if k != 'n_estimators'}
}

xgb_model = xgb.train(
    best_xgb_params,
    dtrain,
    num_boost_round=study_xgb.best_params.get('n_estimators', 300),
    evals=[(dval, 'val')],
    early_stopping_rounds=30,
    verbose_eval=False,
)

y_prob_xgb = xgb_model.predict(dval)
xgb_metrics = {
    'model': 'XGBoost',
    'AUC-ROC': roc_auc_score(y_val, y_prob_xgb),
    'KS': ks_statistic(y_val, y_prob_xgb),
    'Gini': 2 * roc_auc_score(y_val, y_prob_xgb) - 1,
    'Brier': brier_score_loss(y_val, y_prob_xgb),
}
print(xgb_metrics)

## 4. LightGBM con early stopping

LightGBM usa histogramas para un entrenamiento más rápido. Early stopping sobre validación evita overfitting sin necesidad de tuning exhaustivo.

In [ ]:
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val   = lgb.Dataset(X_val,   label=y_val, reference=lgb_train)

lgb_params = {
    'objective':        'binary',
    'metric':           'auc',
    'is_unbalance':     True,
    'learning_rate':    0.05,
    'num_leaves':       63,
    'max_depth':        -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'verbose':          -1,
    'seed':             42,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=-1),
]

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_val],
    callbacks=callbacks,
)

y_prob_lgb = lgb_model.predict(X_val)
lgb_metrics = {
    'model': 'LightGBM',
    'AUC-ROC': roc_auc_score(y_val, y_prob_lgb),
    'KS': ks_statistic(y_val, y_prob_lgb),
    'Gini': 2 * roc_auc_score(y_val, y_prob_lgb) - 1,
    'Brier': brier_score_loss(y_val, y_prob_lgb),
}
print(f'Best iteration: {lgb_model.best_iteration}')
print(lgb_metrics)

## 5. CatBoost

CatBoost maneja internamente las variables numéricas con binning simétrico. No requiere encoding previo y es robusto a outliers.

In [ ]:
cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    auto_class_weights='Balanced',
    eval_metric='AUC',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=False,
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
)

y_prob_cat = cat_model.predict_proba(X_val)[:, 1]
cat_metrics = {
    'model': 'CatBoost',
    'AUC-ROC': roc_auc_score(y_val, y_prob_cat),
    'KS': ks_statistic(y_val, y_prob_cat),
    'Gini': 2 * roc_auc_score(y_val, y_prob_cat) - 1,
    'Brier': brier_score_loss(y_val, y_prob_cat),
}
print(f'Best iteration: {cat_model.best_iteration_}')
print(cat_metrics)

## 6. Tabla comparativa final

Cargamos los resultados baseline de la Fase 3 y los comparamos con los modelos avanzados.

In [ ]:
experiments = json.loads((MODELS / 'experiments.json').read_text())
baseline_results = pd.DataFrame(experiments['baseline']['results']['AUC-ROC'],
                                 index=['AUC-ROC']).T.reset_index().rename(columns={'index': 'model'})
for col in ['KS', 'Gini', 'Brier']:
    baseline_results[col] = pd.DataFrame(experiments['baseline']['results'][col], index=[col]).T.values

advanced_results = pd.DataFrame([xgb_metrics, lgb_metrics, cat_metrics])
all_results = pd.concat([baseline_results, advanced_results], ignore_index=True)
all_results = all_results.set_index('model').sort_values('AUC-ROC', ascending=False)

display(all_results.round(4).style
        .background_gradient(cmap='Blues', subset=['AUC-ROC', 'KS', 'Gini'])
        .background_gradient(cmap='RdYlGn_r', subset=['Brier']))

## 7. Curvas ROC — todos los modelos avanzados

In [ ]:
advanced_probs = {
    'XGBoost':  y_prob_xgb,
    'LightGBM': y_prob_lgb,
    'CatBoost': y_prob_cat,
}

fig, ax = plt.subplots(figsize=(8, 6))
colors_map = ['steelblue', 'tomato', 'green']

for (name, y_prob), color in zip(advanced_probs.items(), colors_map):
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC — Modelos Avanzados')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / '12_roc_advanced_models.png', bbox_inches='tight')
plt.show()

## 8. Calibración de probabilidades

Un modelo con AUC alto puede estar mal calibrado (probabilidades alejadas de la frecuencia real).  
Comparamos calibración raw vs Platt Scaling vs Isotonic Regression.

In [ ]:
# Usamos el mejor modelo avanzado para calibración
best_advanced_name = all_results.index[0]
print(f'Modelo a calibrar: {best_advanced_name}')

# Wrapper sklearn para XGBoost nativo (necesario para CalibratedClassifierCV)
from xgboost import XGBClassifier

best_xgb_sklearn = XGBClassifier(
    **{k: v for k, v in study_xgb.best_params.items() if k != 'n_estimators'},
    n_estimators=study_xgb.best_params.get('n_estimators', 300),
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    verbosity=0,
)
best_xgb_sklearn.fit(X_train, y_train)

cal_platt    = CalibratedClassifierCV(best_xgb_sklearn, method='sigmoid',  cv='prefit')
cal_isotonic = CalibratedClassifierCV(best_xgb_sklearn, method='isotonic', cv='prefit')
cal_platt.fit(X_val, y_val)
cal_isotonic.fit(X_val, y_val)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')

for model_obj, label, color in [
    (best_xgb_sklearn, 'XGBoost raw',   'steelblue'),
    (cal_platt,        'Platt Scaling',  'tomato'),
    (cal_isotonic,     'Isotonic Reg.',  'green'),
]:
    prob_pos = model_obj.predict_proba(X_val)[:, 1]
    # Calibración en test para evaluación honesta
    prob_test = model_obj.predict_proba(X_test)[:, 1]
    fraction_pos, mean_pred = calibration_curve(y_test, prob_test, n_bins=10)
    ax.plot(mean_pred, fraction_pos, marker='o', label=label, color=color)

ax.set_xlabel('Probabilidad predicha')
ax.set_ylabel('Fracción de positivos reales')
ax.set_title('Curva de calibración (reliability diagram)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / '13_calibration_curve.png', bbox_inches='tight')
plt.show()

## 9. Selección del modelo campeón y serialización

In [ ]:
# El campeón final considera AUC + calibración
# Usamos LightGBM o XGBoost calibrado según resultados — ajustar según tabla
champion_name = all_results['AUC-ROC'].idxmax()
champion_probs = advanced_probs[champion_name] if champion_name in advanced_probs else None

champion_map = {
    'XGBoost':  (best_xgb_sklearn, 'champion_xgb.joblib'),
    'LightGBM': (lgb_model,        'champion_lgb.joblib'),
    'CatBoost': (cat_model,        'champion_catboost.joblib'),
}

for name, (model, fname) in champion_map.items():
    joblib.dump(model, MODELS / fname)

# Registro
experiments['advanced'] = {
    'champion': champion_name,
    'xgb_best_params': study_xgb.best_params,
    'xgb_best_auc': study_xgb.best_value,
    'results': all_results.round(4).to_dict(),
}
(MODELS / 'experiments.json').write_text(json.dumps(experiments, indent=2))

print(f'Modelo campeón: {champion_name}')
print(f'AUC-ROC en validación: {all_results.loc[champion_name, "AUC-ROC"]:.4f}')
print('Modelos serializados en models/saved/')

## Resumen Fase 4

**Modelos entrenados:** XGBoost (Optuna 100 trials), LightGBM (early stopping), CatBoost  
**Calibración:** Platt Scaling e Isotonic Regression comparadas en reliability diagram  

**Objetivos del proyecto:**

| Métrica | Mínimo | Objetivo |
|---|---|---|
| AUC-ROC | 0.78 | > 0.82 |
| KS | 0.35 | > 0.45 |
| Gini | 0.56 | > 0.64 |
| Brier | < 0.10 | < 0.07 |

**Próximo paso:** `05_scorecard_woe_iv.ipynb` — construcción de scorecard bancaria